# 02 · Safety and capability across lambda

Hours 17-21. Family B (100 JailbreakBench behaviours, refusal heuristic plus an
optional Llama-Guard-class judge) and Family C (CE loss, MMLU, TruthfulQA MC1,
optionally GSM8K).

No harmful completion is ever reproduced in the paper: raw generations stay in
the local JSONL, and only aggregate scores are exported.

In [ ]:
#@title Clone the repo and install dependencies { display-mode: "form" }
# Colab: paste a GitHub PAT with repo:read scope. It is used only for the clone
# and is not written to disk.
import os, subprocess, sys, getpass, pathlib

REPO   = "sagnikc395/apart-mind-digital-mind"  #@param {type:"string"}
BRANCH = "main"                                 #@param {type:"string"}
WORKDIR = "/content"

if pathlib.Path("/content").exists():
    token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub token (blank if public): ")
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    dest = pathlib.Path(WORKDIR) / REPO.split("/")[-1]
    if dest.exists():
        subprocess.run(["git", "-C", str(dest), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", url, str(dest)], check=True)
    os.chdir(dest)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "torch", "transformers>=4.44", "accelerate", "datasets", "matplotlib"], check=True)
else:
    os.chdir(pathlib.Path.cwd())  # already inside the repo, e.g. running locally

sys.path.insert(0, str(pathlib.Path.cwd() / "src"))
print("cwd:", os.getcwd())


In [ ]:
#@title Persist results to Drive (survives a session kill)
import os, pathlib

RESULTS = "/content/drive/MyDrive/alignment_tax_results"  #@param {type:"string"}
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    RESULTS = str(pathlib.Path.cwd() / "results")
    print("no Drive; writing to", RESULTS, f"({exc})")
os.environ["ALIGNMENT_TAX_RESULTS"] = RESULTS
pathlib.Path(RESULTS).mkdir(parents=True, exist_ok=True)
print("results ->", RESULTS)


In [ ]:
from pathlib import Path
import os
from alignment_tax.config import RunConfig
from alignment_tax import pipeline

results = Path(os.environ.get("ALIGNMENT_TAX_RESULTS", "results"))
cfg = RunConfig.load(next(results.rglob("run_config.json")))
cfg.results_dir = results
hm = pipeline.load_model(cfg)
rd = pipeline.stage_direction(hm, cfg)

In [ ]:
#@title Safety sweep (500 generations at the default grid)
USE_GUARD = False  #@param {type:"boolean"}
pipeline.stage_safety(hm, cfg, rd.vector, use_guard=USE_GUARD)

In [ ]:
#@title Capability sweep
cfg.evals.run_gsm8k = False  #@param {type:"boolean"}
pipeline.stage_capability(hm, cfg, rd.vector)

In [ ]:
import json
print(json.dumps(json.loads(cfg.artifact("capability.json").read_text()), indent=2))